In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
sns.set()

In [ ]:
import datetime as dt

In [ ]:
period = 5
end = dt.date(2025,3,3)
start = end - dt.timedelta(days=period*365)

In [ ]:
print(start,end)

In [ ]:
symbol = "EURUSD=X"

In [ ]:
data = yf.download(tickers = symbol,start=start,end=end,auto_adjust=True)["Close"]

In [ ]:
data.head()

In [ ]:
data.column_names = None

In [ ]:
data.rename({"EURUSD=X":"Price"},axis=1,inplace=True)

In [ ]:
data.columns = data.columns.get_level_values(0)
data.columns.name = None

In [ ]:
data.head()

Mean Reversion

In [ ]:
sma = 30
std = 2

In [ ]:
data.plot(label = "eurusd price series")

In [ ]:
data["SMA"] = data.Price.rolling(sma).mean()

In [ ]:
data.plot()

In [ ]:
data["lower"] = data["SMA"] - data.Price.rolling(sma).std()*std
data["upper"] = data["SMA"] + data.Price.rolling(sma).std()*std

In [ ]:
data.loc["2023"].plot()

In [ ]:
data["Returns"] = np.log(data.Price/data.Price.shift(1))

In [ ]:
data.dropna(inplace=True)

In [ ]:
data["Distance"] = data.Price - data.SMA

In [ ]:
data["Position"] = np.where(data.Price<data.SMA,1,np.nan)
data["Position"] = np.where(data.Price>data.SMA,-1,data["Position"])

In [ ]:
data["Position"] = np.where(data.Distance*data.Distance.shift(1)<0,0,data["Position"])

In [ ]:
data["Position"] = data["Position"].ffill().fillna(0)

In [ ]:
data.Position.value_counts()

In [ ]:
data[["Price","lower","upper","Position"]].loc["2023"].plot(secondary_y="Position")

In [ ]:
data["Strategy"] = data.Position*data.Returns.shift(1)

In [ ]:
data["creturns"] = data.Returns.cumsum().apply(np.exp)
data["cstrategy"] = data.Strategy.cumsum().apply(np.exp)

In [ ]:
data[["creturns","cstrategy"]].loc["2023"].plot()